# Notebook 7 – Binning & Discretization


## Learning objectives
- Explain binning and why continuous values may be converted into categories.
- Apply equal-width, equal-frequency/quantile, and domain-based binning.
- Create age, income, and risk groups.
- Understand information loss, boundary effects, and how to avoid data leakage.

## 1. What is binning?

Binning (also called discretization) converts a continuous numeric feature into a limited number of intervals called **bins**.

Example: exact ages `18, 21, 27, 34, 51` can become categories such as `18–25`, `26–35`, `36–50`, and `51+`.

Continuous → categorical:
- Before: `age = 27`
- After: `age_group = "26-35"`

Binning simplifies values and can make patterns easier to communicate. However, it loses detail: ages 26 and 35 become the same category.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "customer_id": range(1, 11),
    "age": [18, 21, 24, 27, 31, 35, 39, 45, 52, 63],
    "annual_income": [180000, 220000, 260000, 300000, 350000,
                      420000, 500000, 650000, 850000, 1200000],
    "risk_score": [12, 18, 22, 31, 38, 45, 57, 66, 78, 92]
})
df

## 2. Why binning?
- Makes complex numeric patterns easier to explain.
- Can reduce the effect of small measurement fluctuations.
- Can represent known business thresholds (for example, policy risk bands).
- May help some models capture non-linear patterns.

Do not bin automatically. Many models can use continuous values directly, and binning can reduce predictive information.

## 3. Equal-width binning
Equal-width binning divides the full numeric range into bins of the same width.

For range minimum \(a\), maximum \(b\), and \(k\) bins:

\[
\text{Bin width} = \frac{b-a}{k}
\]

If ages range from 18 to 63 and we use 3 bins, each bin has approximately the same numeric width. The number of records in each bin may differ.

In [ ]:
df["age_equal_width"] = pd.cut(
    df["age"],
    bins=3,
    labels=["Low age", "Middle age", "High age"],
    include_lowest=True
)
df[["age", "age_equal_width"]]

**Expected output:** The age range is split into three intervals of equal width. The categories are ordered from low to high age; exact boundaries are calculated by `pd.cut`.

## 4. Equal-frequency / quantile binning
Equal-frequency binning aims to place approximately the same number of observations in each bin. Quantile binning is a common implementation.

`q=4` creates quartile groups. If many values are tied, duplicate bin edges can occur; `duplicates="drop"` allows fewer bins rather than failing.

In [ ]:
df["income_quantile"] = pd.qcut(
    df["annual_income"],
    q=4,
    labels=["Q1 - Low", "Q2", "Q3", "Q4 - High"],
    duplicates="drop"
)
df[["annual_income", "income_quantile"]]

**Expected output:** The customers are divided into income groups based on the distribution. With no tied values in this sample, each quartile contains roughly the same number of rows (some bins can differ by one row).

## 5. Compare bin counts

In [ ]:
print("Equal-width age counts:")
print(df["age_equal_width"].value_counts().sort_index())

print("\\nQuantile income counts:")
print(df["income_quantile"].value_counts().sort_index())

Equal-width bins have equal numeric widths, not equal row counts. Quantile bins aim for similar row counts, not equal numeric widths.

## 6. Domain-based binning
Domain-based bins use meaningful business or expert-defined thresholds rather than automatically calculating boundaries.

Examples:
- Age bands used by a specific insurance product.
- Income bands defined by a bank's internal policy.
- Risk levels defined by an approved risk framework.

The thresholds below are illustrative only. Real thresholds must come from the relevant business rules or domain experts.

In [ ]:
age_bins = [0, 25, 35, 50, np.inf]
age_labels = ["18-25", "26-35", "36-50", "51+"]

df["age_group"] = pd.cut(
    df["age"], bins=age_bins, labels=age_labels,
    right=True, include_lowest=True
)

income_bins = [0, 300000, 600000, 1000000, np.inf]
income_labels = ["Low", "Lower-middle", "Upper-middle", "High"]

df["income_group"] = pd.cut(
    df["annual_income"], bins=income_bins,
    labels=income_labels, include_lowest=True
)

risk_bins = [0, 30, 60, 80, 100]
risk_labels = ["Low risk", "Moderate risk", "High risk", "Very high risk"]

df["risk_category"] = pd.cut(
    df["risk_score"], bins=risk_bins,
    labels=risk_labels, include_lowest=True
)

df[["age", "age_group", "annual_income", "income_group",
    "risk_score", "risk_category"]]

## 7. Visualize the groups

In [ ]:
import matplotlib.pyplot as plt

df["age_group"].value_counts(sort=False).plot(kind="bar")
plt.title("Customer Count by Age Group")
plt.xlabel("Age group")
plt.ylabel("Number of customers")
plt.xticks(rotation=0)
plt.show()

**Interpretation:** The chart shows how many customers fall into each age band. It describes this sample; it does not prove that age causes any outcome.

## 8. Advantages and limitations

| Advantages | Limitations |
|---|---|
| Easier interpretation and reporting | Loses exact numeric information |
| Can reduce sensitivity to small fluctuations | Values near a boundary may be treated very differently |
| Supports business-defined categories | Thresholds can be subjective or outdated |
| Can help represent non-linear patterns | May reduce model performance |
| Can simplify some visualizations | Quantile bins can change across datasets |

## 9. Important implementation cautions
- Check the minimum and maximum values before choosing boundaries.
- Make sure every valid value falls into a bin; otherwise values may become missing.
- Document whether interval boundaries are inclusive or exclusive.
- Treat categories as ordered only when the order has real meaning.
- For production ML, learn data-driven bin boundaries on the training set only and apply the same boundaries to validation/test data. Do not calculate bins using the full dataset before splitting.
- Do not create bins using the target variable unless the method is explicitly supervised and is fitted correctly within cross-validation.

